# 04_205 · Fine-tuning reanudable de Qwen con cuatro daños

> **Espacio reservado para la sesión iniciada como `04_7`.** Mientras ese kernel siga entrenando, no ejecute este cuaderno en paralelo. Los checkpoints y resultados externos son compartidos.

Este cuaderno entrena **solamente Qwen3-0.6B-Base con LoRA**. Las cuatro salidas operativas son `RACISMO_DISCRIMINACION`, `ACOSO_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `ACOSO_AMENAZA` es la unión de las antiguas `ACOSO_PERSONAL` y `AMENAZA_DIRECTA`; `SEGURO` se deriva cuando ninguna salida supera su umbral.

Las etiquetas finas y los flags transversales se usan sólo como supervisión auxiliar multitararea. No son variables de entrada, no agregan categorías operativas y las etiquetas finas ausentes quedan enmascaradas. El cuaderno guarda resultados por época y checkpoints completos reanudables durante cada época.

El arranque funciona tanto localmente como en Colab. En local recupera el checkpoint compartido que haya dejado `04_7` o este mismo cuaderno; en Colab usa la instantánea verificada de ese checkpoint sincronizada en Google Drive. `run_finetuning(..., resume=True, force_restart=False)` detecta y reanuda ese estado automáticamente.

In [ ]:
from pathlib import Path
from importlib.util import find_spec
import importlib, inspect
import hashlib, json, os, shutil, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

REPO_URL = 'https://github.com/lkoc/Trabajo_PLN-MIA-Grupo4.git'
GIT_COMMIT = 'ce33b5efd797eef0809e9ff8694c90220cd81e9d'
PROJECT_NAME, DRIVE_BUNDLE_NAME, NEEDS_PEFT = 'Trabajo_PLN-MIA-Grupo4', 'PLN_colab_04_artifacts', True

def _bootstrap_04_20x():
    in_colab = find_spec('google.colab') is not None
    if not in_colab:
        start = Path.cwd().resolve()
        root = next((p for p in (start, *start.parents) if (p / 'scripts_auxiliares').is_dir()), None)
        if root is None: raise FileNotFoundError('No se encontró la raíz local del proyecto.')
        return root, False, root, 'working-tree-local'
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    artifacts = Path('/content/drive/MyDrive') / DRIVE_BUNDLE_NAME
    manifest_path = artifacts / 'MANIFIESTO_ARTEFACTOS_04_20X.json'
    if not manifest_path.is_file(): raise FileNotFoundError('Falta el bundle de Drive; ejecute sincronizar_04_20x_google_drive.ps1 en Windows.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    # Durante una extensión se archiva temporalmente finetuning.json para
    # que el código Colab fijado cargue el checkpoint. Si el runtime se
    # reinicia, restauramos el resultado histórico más reciente y coherente.
    training_relative = 'resultados/metricas/qwen3_06b_lora_acoso_amenaza_4/finetuning.json'
    training_path = artifacts / training_relative
    resume_pointer = artifacts / 'modelos/qwen3_06b_lora_acoso_amenaza_4/resume_pointer.json'
    if not training_path.is_file() and resume_pointer.is_file():
        pointer = json.loads(resume_pointer.read_text(encoding='utf-8'))
        archives = sorted(training_path.parent.glob('finetuning_hasta_epoca_*.json'), reverse=True)
        restored = None
        for candidate in archives:
            try:
                candidate_result = json.loads(candidate.read_text(encoding='utf-8'))
                suffix_epoch = int(candidate.stem.rsplit('_', 1)[-1])
                coherent = (
                    candidate_result.get('status') == 'completed'
                    and int(candidate_result.get('epochs_completed', -1)) == suffix_epoch
                    and len(candidate_result.get('history', [])) == suffix_epoch
                    and candidate_result.get('dataset_sha256') == pointer.get('dataset_sha256')
                )
            except Exception:
                coherent = False
            if coherent:
                restored = candidate
                break
        if restored is not None:
            digest = hashlib.sha256(restored.read_bytes()).hexdigest()
            shutil.copy2(restored, training_path)
            if hashlib.sha256(training_path.read_bytes()).hexdigest() != digest:
                raise RuntimeError('Falló la restauración verificada de finetuning.json.')
            print(f'Restaurado {restored.name} para reanudar de forma segura.')
    missing = [r['path'] for r in manifest['files'] if not (artifacts / r['path']).is_file()]
    if missing: raise FileNotFoundError('Bundle incompleto en Drive:\n' + '\n'.join(missing))
    root = Path('/content') / PROJECT_NAME
    if not (root / '.git').is_dir():
        if root.exists(): raise RuntimeError(f'Ruta no administrada ya existente: {root}')
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(root)], check=True)
    current = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    if current != GIT_COMMIT:
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', GIT_COMMIT], check=True)
    subprocess.run(['git', '-C', str(root), 'sparse-checkout', 'set', 'scripts_auxiliares'], check=True)
    subprocess.run(['git', '-C', str(root), 'checkout', '--detach', GIT_COMMIT], check=True)
    for name in ('datos', 'modelos', 'resultados'):
        local, target = root / name, artifacts / name
        target.mkdir(parents=True, exist_ok=True)
        if local.is_symlink() and local.resolve() == target.resolve(): continue
        if local.is_symlink(): local.unlink()
        elif local.exists(): raise RuntimeError(f'Ruta de artefactos no administrada: {local}')
        local.symlink_to(target, target_is_directory=True)
    return root, True, artifacts, GIT_COMMIT

ROOT, IN_COLAB, ARTIFACT_ROOT, CODE_VERSION = _bootstrap_04_20x()
if IN_COLAB:
    packages = {'transformers': 'transformers>=4.51,<6', 'sklearn': 'scikit-learn>=1.4', 'peft': 'peft>=0.15,<1'}
    missing_packages = [p for m, p in packages.items() if find_spec(m) is None]
    if missing_packages: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=True)
    # TorchAO es opcional para este LoRA float32. Algunas imágenes de Colab
    # incluyen una versión antigua que PEFT rechaza incluso sin usar cuantización.
    try:
        from peft import import_utils as _peft_import_utils
        _peft_import_utils.is_torchao_available()
    except ImportError as error:
        if 'incompatible version of torchao' not in str(error): raise
        try:
            from importlib.metadata import version as _package_version
            _torchao_version = _package_version('torchao')
        except Exception:
            _torchao_version = 'desconocida'
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
        for _module in tuple(sys.modules):
            if _module == 'torchao' or _module.startswith('torchao.') or _module == 'peft' or _module.startswith('peft.'):
                sys.modules.pop(_module, None)
        importlib.invalidate_caches()
        if find_spec('torchao') is not None: raise RuntimeError('TorchAO incompatible continúa importable; reinicie el runtime.')
        print(f'TorchAO {_torchao_version} incompatible retirado; PEFT usará LoRA estándar.')
os.environ['PLN_PROJECT_ROOT'], os.environ['PLN_ARTIFACT_ROOT'] = str(ROOT), str(ARTIFACT_ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from scripts_auxiliares import entrenar_qwen_acoso_amenaza as qwen4

print('Entorno:', 'Google Colab híbrido' if IN_COLAB else 'local')
print('Código:', ROOT, CODE_VERSION)
print('Artefactos persistentes:', ARTIFACT_ROOT)
print('Modelo:', qwen4.MODEL_SPEC.model_id)
print('Objetivos primarios:', qwen4.TARGET_LABELS)
print('SEGURO derivada:', True)
print('Dispositivo disponible:', qwen4.device())
print('Checkpoint recuperable (04_7/04_205):', qwen4.resume_status())

## 1. Dataset congelado, unión de etiquetas y auditoría auxiliar

Se reutilizan exactamente el dataset 4:1 y los splits `train/validation/test` creados por `04_2`. Primero se submuestreó `SEGURO` sin retirar daños y después se particionó; este cuaderno no vuelve a muestrear ni modifica los splits. La unión se calcula fila por fila como `max(ACOSO_PERSONAL, AMENAZA_DIRECTA)`.

In [ ]:
frames, dataset_audit = qwen4.load_frames()
display(qwen4.dataset_summary(dataset_audit))

coverage = pd.DataFrame(dataset_audit['auxiliary_supervision']['split_coverage']).T
display(coverage[['fine_rows_available', 'fine_rows_masked', 'fine_coverage', 'flag_rows_available']])

print('SHA-256 dataset:', dataset_audit['dataset_sha256'])
print('Fingerprint de entrenamiento:', dataset_audit['training_fingerprint_sha256'])
print('Prompt operativo:', dataset_audit['prompt_operational'])

In [ ]:
summary = qwen4.dataset_summary(dataset_audit).set_index('split')
ax = summary[qwen4.TARGET_LABELS].plot.bar(figsize=(11, 5), width=0.8)
ax.set_title('Positivos por objetivo primario y partición')
ax.set_ylabel('Chunks positivos')
ax.set_xlabel('Partición')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Metodología

La tarea primaria es multietiqueta con cuatro logits. Se minimiza entropía cruzada binaria ponderada, y se añaden pérdidas auxiliares de etiquetas finas (peso 0,20) y flags transversales (peso 0,15). La pérdida fina sólo se calcula para los chunks con anotación fina recuperable; una anotación ausente **no** se convierte en una colección de ceros. Las cabezas auxiliares se descartan para la decisión operativa. Este diseño busca que representaciones compartidas aprendan distinciones útiles sin cambiar la taxonomía de producción (Ruder, 2017).

El prompt operativo **no es necesario como entrada del modelo**. Se registra con hash como procedencia de las definiciones usadas al etiquetar. Inyectarlo en cada texto crearía una condición distinta de la inferencia real.

Qwen se ajusta mediante LoRA (Hu et al., 2022) y se conserva un adaptador independiente por época. Tras completar las cuatro, se toman las dos épocas con mayor PR-AUC macro de daño en validación; cada una recibe su propia calibración sigmoide y umbrales. Ambas se comparan con una política de revisión ajustada al mismo objetivo de 95% de recall, todavía sólo en validación (Niculescu-Mizil & Caruana, 2005). Recién después de fijar el ganador operativo se abre test una sola vez para la estimación final; sus resultados no retroalimentan la selección.

## 3. Estado reanudable

Se guarda un checkpoint cada 250 pasos del optimizador, alternando dos directorios para reducir el riesgo de corrupción. Incluye adaptador, optimizador, scheduler, estados aleatorios, época y siguiente lote. También se guardan `best_adapter`, `last_adapter`, logits y métricas por época. Si VS Code, el kernel o la máquina se interrumpen, vuelva a ejecutar desde la primera celda y después la celda de entrenamiento: continuará desde el último checkpoint compatible.

In [ ]:
status = qwen4.resume_status()
if qwen4.TRAINING_RESULT_PATH.is_file() and qwen4.RESUME_POINTER_PATH.is_file():
    _result_status = json.loads(qwen4.TRAINING_RESULT_PATH.read_text(encoding='utf-8'))
    _pointer_status = json.loads(qwen4.RESUME_POINTER_PATH.read_text(encoding='utf-8'))
    if int(_result_status.get('epochs_completed', 0)) < 4 and int(_pointer_status.get('epoch', 0)) > 4:
        status = {**status, 'status': 'force_extendable', 'next_epoch': int(_result_status['epochs_completed']) + 1}
display(status)

## 4. Entrenamiento de Qwen

Esta es la única celda que entrena un modelo. La época 3 activó la parada temprana por una diferencia mínima de PR-AUC, pero el checkpoint reanudable conserva sus pesos, Adam, scheduler y estados aleatorios. En esta ejecución se anula esa parada **sólo para completar obligatoriamente la época 4**, sin reiniciar ni repetir las tres anteriores. También se preservan adaptadores independientes por época para poder calibrar los dos mejores. Si se interrumpe, ejecute otra vez con `resume=True`. No use `force_restart=True`, pues iniciaría deliberadamente desde cero.

In [ ]:
import importlib, inspect
qwen4 = importlib.reload(qwen4)  # evita definiciones antiguas conservadas por el kernel
print('Módulo Qwen recargado:', qwen4.__file__)

TARGET_MAX_EPOCHS = 4
qwen4.MAX_EPOCHS = TARGET_MAX_EPOCHS

# Compatibilidad CPU → GPU: torch.load(map_location=cuda) también mueve
# el RNG guardado, pero torch.set_rng_state exige un ByteTensor en CPU.
def _restore_rng_state_portable(state):
    qwen4.random.setstate(state['python'])
    qwen4.np.random.set_state(state['numpy'])
    cpu_rng = state['torch']
    if isinstance(cpu_rng, qwen4.torch.Tensor):
        cpu_rng = cpu_rng.detach().to(device='cpu', dtype=qwen4.torch.uint8)
    else:
        cpu_rng = qwen4.torch.as_tensor(cpu_rng, dtype=qwen4.torch.uint8, device='cpu')
    qwen4.torch.set_rng_state(cpu_rng)
    if qwen4.torch.cuda.is_available() and state.get('cuda') is not None:
        cuda_rng = [
            value.detach().to(device='cpu', dtype=qwen4.torch.uint8)
            if isinstance(value, qwen4.torch.Tensor)
            else qwen4.torch.as_tensor(value, dtype=qwen4.torch.uint8, device='cpu')
            for value in state['cuda']
        ]
        qwen4.torch.cuda.set_rng_state_all(cuda_rng)
qwen4._restore_rng_state = _restore_rng_state_portable

def _preserve_epoch_adapter(source, epoch):
    source = Path(source)
    target = qwen4.MODEL_DIR / 'epoch_adapters' / f'epoch_{int(epoch):02d}'
    if target.exists():
        if not (target / 'training_state.json').is_file() or not (target / 'adapter_model.safetensors').is_file():
            raise FileNotFoundError(f'Adaptador histórico incompleto: {target}')
        saved = json.loads((target / 'training_state.json').read_text(encoding='utf-8'))
        if int(saved['epoch']) != int(epoch): raise RuntimeError(f'Adaptador histórico incoherente: {target}')
        return target
    state_path = source / 'training_state.json'
    weights_path = source / 'adapter_model.safetensors'
    if not state_path.is_file() or not weights_path.is_file(): raise FileNotFoundError(f'Adaptador incompleto: {source}')
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if int(state['epoch']) != int(epoch): raise RuntimeError(f'{source} no corresponde a la época {epoch}.')
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)
    if qwen4.sha256_file(target / 'training_state.json') != qwen4.sha256_file(state_path):
        raise RuntimeError(f'Falló la copia verificada de {source}.')
    if qwen4.sha256_file(target / 'adapter_model.safetensors') != qwen4.sha256_file(weights_path):
        raise RuntimeError(f'Falló la copia verificada de los pesos de {source}.')
    return target

# Conserva los candidatos ya entrenados antes de que last_adapter cambie.
if qwen4.TRAINING_RESULT_PATH.is_file():
    previous_result = json.loads(qwen4.TRAINING_RESULT_PATH.read_text(encoding='utf-8'))
    completed_epochs = int(previous_result.get('epochs_completed', 0))
    if completed_epochs:
        _preserve_epoch_adapter(qwen4.MODEL_DIR / 'best_adapter', int(previous_result['best_epoch']))
        _preserve_epoch_adapter(qwen4.MODEL_DIR / 'last_adapter', completed_epochs)

# Early stopping dejó epoch=5 tras terminar la época 3. Se carga ese mismo
# checkpoint y sólo se corrige el próximo epoch a 4; no se reinician pesos,
# Adam, scheduler ni RNG. También cubre una interrupción dentro de época 4.
_original_load_resume_checkpoint = qwen4._load_resume_checkpoint
def _load_resume_for_forced_epoch4(*args, **kwargs):
    loaded = _original_load_resume_checkpoint(*args, **kwargs)
    if loaded is None: return None
    model, state, optimizer_state = loaded
    completed_history = len(state.get('history', []))
    if completed_history < TARGET_MAX_EPOCHS and int(state['epoch']) > TARGET_MAX_EPOCHS:
        state.update(epoch=completed_history + 1, completed_batches=0, cumulative_loss=0.0,
                     seen=0, optimizer_steps_in_epoch=0, stale_epochs=0)
        print(f'Early stopping anulado: se reanudará la época {state["epoch"]}/{TARGET_MAX_EPOCHS}.')
    return model, state, optimizer_state
qwen4._load_resume_checkpoint = _load_resume_for_forced_epoch4

# El commit fijo de Colab puede considerar finetuning.json como terminal.
# Lo archivamos con hash verificado y lo retiramos sólo durante la extensión.
if qwen4.TRAINING_RESULT_PATH.is_file():
    previous_result = json.loads(qwen4.TRAINING_RESULT_PATH.read_text(encoding='utf-8'))
    completed_epochs = int(previous_result.get('epochs_completed', 0))
    if completed_epochs < TARGET_MAX_EPOCHS:
        if not qwen4.RESUME_POINTER_PATH.is_file(): raise FileNotFoundError('Falta resume_pointer.json; no se reiniciará desde cero.')
        archive = qwen4.TRAINING_RESULT_PATH.with_name(f'finetuning_hasta_epoca_{completed_epochs:02d}.json')
        if archive.exists() and qwen4.sha256_file(archive) != qwen4.sha256_file(qwen4.TRAINING_RESULT_PATH):
            raise RuntimeError(f'Ya existe un archivo histórico diferente: {archive}')
        if not archive.exists(): shutil.copy2(qwen4.TRAINING_RESULT_PATH, archive)
        if qwen4.sha256_file(archive) != qwen4.sha256_file(qwen4.TRAINING_RESULT_PATH):
            raise RuntimeError('No se pudo verificar la copia del resultado previo.')
        qwen4.TRAINING_RESULT_PATH.unlink()
        print(f'Resultado previo preservado en {archive}; se reutilizará el checkpoint existente.')

training_kwargs = dict(frames=frames, resume=True, force_restart=False)
if 'force_complete_max_epochs' in inspect.signature(qwen4.run_finetuning).parameters:
    training_kwargs['force_complete_max_epochs'] = True
training_result = qwen4.run_finetuning(**training_kwargs)
if int(training_result['epochs_completed']) != TARGET_MAX_EPOCHS:
    raise RuntimeError(f'La época 4 no quedó completa: {training_result["epochs_completed"]} épocas registradas.')
_preserve_epoch_adapter(qwen4.MODEL_DIR / 'last_adapter', TARGET_MAX_EPOCHS)
display({
    'status': training_result['status'],
    'max_epochs': training_result['max_epochs'],
    'epochs_completed': training_result['epochs_completed'],
    'best_epoch': training_result['best_epoch'],
    'best_validation_damage_pr_auc_macro': training_result['best_validation_damage_pr_auc_macro'],
    'adapter': training_result['adapter'],
})

In [ ]:
history_path = qwen4.METRICS_DIR / 'historial.csv'
history = pd.read_csv(history_path)
display(history)
ax = history.plot(x='epoch', y=['damage_pr_auc_macro', 'damage_f1_macro', 'any_damage_recall'], marker='o', figsize=(10, 5))
ax.set_ylim(0, 1)
ax.set_title('Evolución en validación')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Calibración y selección de los dos mejores checkpoints (sólo validación)

Los dos candidatos se determinan por PR-AUC macro de daño en el historial de validación. Cada checkpoint recibe su propio calibrador sigmoide, umbrales por etiqueta y corte de revisión humana ajustado al mismo objetivo de recall (95%). El ganador operativo minimiza la tasa de revisión entre quienes cumplen el objetivo; los desempates favorecen mayor precisión, menos falsos negativos y mayor PR-AUC. **Esta celda no carga ni consulta `test`.**

In [ ]:
RECALL_TARGET = 0.95
SELECTION_PATH = qwen4.METRICS_DIR / 'seleccion_operativa_validacion.json'
y_validation = qwen4.four_targets(frames['validation']).astype(qwen4.np.int8)
history_records = training_result['history']
if len(history_records) < 2: raise RuntimeError('Se requieren al menos dos épocas completas.')
top_two = sorted(
    history_records,
    key=lambda row: (float(row['damage_pr_auc_macro']), -int(row['epoch'])),
    reverse=True,
)[:2]

candidate_results = []
for training_row in top_two:
    epoch = int(training_row['epoch'])
    adapter_dir = qwen4.MODEL_DIR / 'epoch_adapters' / f'epoch_{epoch:02d}'
    state_path = adapter_dir / 'training_state.json'
    weights_path = adapter_dir / 'adapter_model.safetensors'
    logits_path = qwen4.METRICS_DIR / f'validation_logits_primary_epoch_{epoch:02d}.npy'
    if not state_path.is_file() or not weights_path.is_file() or not logits_path.is_file():
        raise FileNotFoundError(f'Faltan artefactos del candidato de época {epoch}.')
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if int(state['epoch']) != epoch: raise RuntimeError(f'Adaptador incoherente para época {epoch}.')
    validation_logits = qwen4.np.load(logits_path)
    if validation_logits.shape != y_validation.shape or not qwen4.np.isfinite(validation_logits).all():
        raise RuntimeError(f'Logits de validación inválidos para época {epoch}: {validation_logits.shape}.')

    calibrators = qwen4.fit_calibrators(y_validation, validation_logits)
    validation_scores = qwen4.apply_calibrators(calibrators, validation_logits)
    thresholds = qwen4.tune_thresholds(y_validation, validation_scores)
    classification_metrics, _, _ = qwen4.evaluate_scores(y_validation, validation_scores, thresholds)
    cutoff, routing = qwen4.tm.tune_human_alert_cutoff(
        y_validation, validation_scores, thresholds, recall_target=RECALL_TARGET
    )
    calibrator_path = qwen4.METRICS_DIR / f'calibradores_epoch_{epoch:02d}.joblib'
    scores_path = qwen4.METRICS_DIR / f'scores_calibrated_validation_epoch_{epoch:02d}.npy'
    qwen4.joblib.dump(calibrators, calibrator_path)
    qwen4.np.save(scores_path, validation_scores)
    candidate = {
        'epoch': epoch,
        'adapter': str(adapter_dir.relative_to(ROOT)).replace('\\', '/'),
        'adapter_training_state_sha256': qwen4.sha256_file(state_path),
        'adapter_weights_sha256': qwen4.sha256_file(weights_path),
        'validation_logits': str(logits_path.relative_to(ROOT)).replace('\\', '/'),
        'calibrator': str(calibrator_path.relative_to(ROOT)).replace('\\', '/'),
        'validation_scores': str(scores_path.relative_to(ROOT)).replace('\\', '/'),
        'training_validation_damage_pr_auc_macro': float(training_row['damage_pr_auc_macro']),
        'thresholds_selected_on_validation': thresholds.tolist(),
        'classification_validation': classification_metrics,
        'routing_validation_at_target_recall': routing,
        'risk_margin_cutoff': float(cutoff),
    }
    qwen4.tm.write_json(qwen4.METRICS_DIR / f'calibracion_operativa_epoch_{epoch:02d}.json', candidate)
    candidate_results.append(candidate)

eligible = [row for row in candidate_results if row['routing_validation_at_target_recall']['recall'] + 1e-12 >= RECALL_TARGET]
if not eligible: raise RuntimeError('Ninguno de los dos candidatos alcanzó 95% de recall en validación.')
selected = min(
    eligible,
    key=lambda row: (
        row['routing_validation_at_target_recall']['review_rate'],
        -row['routing_validation_at_target_recall']['precision'],
        row['routing_validation_at_target_recall']['false_negatives'],
        -row['training_validation_damage_pr_auc_macro'],
        row['epoch'],
    ),
)
selection = {
    'completed_at': qwen4.tm.now_iso(),
    'selection_partition': 'validation',
    'test_consulted_for_selection': False,
    'recall_target': RECALL_TARGET,
    'candidate_ranking_basis': 'top two epochs by validation damage_pr_auc_macro',
    'selection_rule': 'minimum validation review_rate subject to recall >= 0.95; ties: precision desc, false_negatives asc, PR-AUC desc',
    'selected_epoch': selected['epoch'],
    'selected_adapter': selected['adapter'],
    'candidates': candidate_results,
}
qwen4.tm.write_json(SELECTION_PATH, selection)

candidate_table = pd.DataFrame([{
    'época': row['epoch'],
    'PR-AUC validación': row['training_validation_damage_pr_auc_macro'],
    'recall objetivo': row['routing_validation_at_target_recall']['recall'],
    'precisión revisión': row['routing_validation_at_target_recall']['precision'],
    'tasa revisión': row['routing_validation_at_target_recall']['review_rate'],
    'falsos negativos': row['routing_validation_at_target_recall']['false_negatives'],
    'seleccionado': row['epoch'] == selected['epoch'],
} for row in candidate_results]).sort_values('seleccionado', ascending=False)
display(candidate_table)
display(Markdown(f'**Checkpoint seleccionado sólo con validación:** época {selected["epoch"]}. Test permanece sin consultar.'))

## 6. Evaluación final del checkpoint ya seleccionado

A partir de aquí sí se abre `test`, una sola vez y después de haber fijado la época, sus calibradores, umbrales y corte de riesgo con validación. Las métricas de test son estimación final, no insumo para cambiar el ganador.

In [ ]:
selection = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
if selection.get('test_consulted_for_selection') is not False:
    raise RuntimeError('La selección no declara aislamiento de test.')
selected = next(row for row in selection['candidates'] if row['epoch'] == selection['selected_epoch'])
selected_adapter_dir = ROOT / selected['adapter']
if qwen4.sha256_file(selected_adapter_dir / 'training_state.json') != selected['adapter_training_state_sha256']:
    raise RuntimeError('El adaptador seleccionado cambió después de la selección.')
if qwen4.sha256_file(selected_adapter_dir / 'adapter_model.safetensors') != selected['adapter_weights_sha256']:
    raise RuntimeError('Los pesos seleccionados cambiaron después de la selección.')

calibrators = qwen4.joblib.load(ROOT / selected['calibrator'])
thresholds = qwen4.np.asarray(selected['thresholds_selected_on_validation'], dtype=float)
cutoff = float(selected['risk_margin_cutoff'])
target_device = qwen4.device()
model = qwen4.load_adapter(selected_adapter_dir, target_device)
test_logits_all = qwen4.predict_logits(
    model, qwen4.evaluation_loader(frames['test'], qwen4.tokenizer()),
    f'Qwen época {selected["epoch"]} · test final',
)
test_logits = test_logits_all[:, :qwen4.PRIMARY_OUTPUTS]
test_scores = qwen4.apply_calibrators(calibrators, test_logits)
y_test = qwen4.four_targets(frames['test']).astype(qwen4.np.int8)
test_metrics, test_report, _ = qwen4.evaluate_scores(y_test, test_scores, thresholds)
test_margin = qwen4.np.max(test_scores - thresholds, axis=1)
test_routing = qwen4.tm._binary_routing_metrics(
    y_test.astype(bool).any(axis=1), test_margin >= cutoff
)
test_routing['risk_margin_cutoff_selected_on_validation'] = cutoff
test_routing['validation_recall_target'] = selection['recall_target']
test_gate = qwen4.tm._human_alert_gate(test_routing)
test_result = {
    'completed_at': qwen4.tm.now_iso(),
    'selected_epoch_fixed_before_test': selected['epoch'],
    'selection_artifact': str(SELECTION_PATH.relative_to(ROOT)).replace('\\', '/'),
    'test_used_for_selection': False,
    'calibration_partition': 'validation',
    'threshold_partition': 'validation',
    'routing_cutoff_partition': 'validation',
    'classification_test': test_metrics,
    'routing_test_at_validation_cutoff': test_routing,
    'human_review_gate': test_gate,
}
TEST_RESULT_PATH = qwen4.METRICS_DIR / 'evaluacion_test_modelo_seleccionado.json'
qwen4.np.save(qwen4.METRICS_DIR / f'logits_test_selected_epoch_{selected["epoch"]:02d}.npy', test_logits)
qwen4.np.save(qwen4.METRICS_DIR / f'scores_test_selected_epoch_{selected["epoch"]:02d}.npy', test_scores)
test_report.to_csv(qwen4.METRICS_DIR / f'reporte_test_selected_epoch_{selected["epoch"]:02d}.csv')
qwen4.tm.write_json(TEST_RESULT_PATH, test_result)
del model
if qwen4.torch.cuda.is_available(): qwen4.torch.cuda.empty_cache()

display(pd.DataFrame([test_metrics]).drop(columns=['category_recall']))
display(pd.DataFrame.from_dict(test_metrics['category_recall'], orient='index', columns=['recall_test']))
display(pd.DataFrame([test_routing])[['recall', 'precision', 'review_rate', 'false_negatives', 'negative_predictive_value']])
display(test_gate)
display(Markdown(
    f'**Selección validación:** `{SELECTION_PATH.relative_to(ROOT)}`  \n'
    f'**Evaluación final:** `{TEST_RESULT_PATH.relative_to(ROOT)}`  \n'
    f'**Adaptador elegido:** `{selected_adapter_dir.relative_to(ROOT)}`'
))

In [ ]:
ordinary = test_result['classification_test']
selective = test_result['routing_test_at_validation_cutoff']
production_ready = bool(test_result['human_review_gate']['passed'])
verdict = (
    '**candidato únicamente a piloto supervisado**'
    if production_ready else
    '**no está listo para moderación autónoma ni para bloqueo o sanción en producción**'
)
display(Markdown(f'''## 7. Conclusión sobre desempeño y uso en producción

Tras completar la época 4, los dos mejores checkpoints se calibraron y compararon al mismo objetivo de 95% de recall usando exclusivamente validación. La época **{selected['epoch']}** fue seleccionada antes de abrir test. En test obtiene PR-AUC macro de daño **{ordinary['damage_pr_auc_macro']:.4f}**, F1 macro **{ordinary['damage_f1_macro']:.4f}** y recall ordinario de cualquier daño **{ordinary['any_damage_recall']:.4f}**.

Con el corte de alto recall fijado en validación, alcanza en test recall **{selective['recall']:.4f}**, precisión **{selective['precision']:.4f}**, tasa de revisión **{selective['review_rate']:.2%}**, VPN **{selective['negative_predictive_value']:.4f}** y deja **{selective['false_negatives']}** falsos negativos. Con estos resultados, el modelo {verdict}. Incluso si supera las puertas numéricas, debe validarse prospectivamente con prevalencia real, gold standard humano, capacidad de revisión, latencia, coste, deriva y desempeño por subgrupos.

La conclusión de test no modifica retrospectivamente el checkpoint seleccionado.
'''))

## Referencias (APA 7)

Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2022). LoRA: Low-rank adaptation of large language models. *International Conference on Learning Representations*. https://openreview.net/forum?id=nZeVKeeFYf9

Niculescu-Mizil, A., & Caruana, R. (2005). Predicting good probabilities with supervised learning. In *Proceedings of the 22nd International Conference on Machine Learning* (pp. 625–632). ACM. https://doi.org/10.1145/1102351.1102430

Qwen Team. (2025). Qwen3 technical report. *arXiv*. https://doi.org/10.48550/arXiv.2505.09388

Ruder, S. (2017). An overview of multi-task learning in deep neural networks. *arXiv*. https://doi.org/10.48550/arXiv.1706.05098